In [ ]:
pip install paho-mqtt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.7 MB/s eta 0:00:00


Program Terbaru 2.0

In [ ]:
import json
import time
import random
from datetime import datetime, timezone, timedelta
import paho.mqtt.client as mqtt

# =====================================================
# TIME CONFIG (WIB UTC+7)
# =====================================================
def get_wib_time():
    wib = timezone(timedelta(hours=7))
    return datetime.now(wib).strftime("%Y-%m-%d %H:%M:%S")

# =====================================================
# MQTT CONFIG
# =====================================================
BROKER = "broker.emqx.io"
PORT   = 1883

DELAY_ANTAR_NODE  = 0.5   # detik — jeda antar publish per node
JEDA_ANTAR_SIKLUS = 17.5  # detik — jeda setelah semua node selesai
# Total per siklus: 25 × 0.5s + 17.5s = ~30 detik

client = mqtt.Client(
    client_id=f"wanara_sim_{random.randint(100000, 999999)}",
    clean_session=True
)

mqtt_connected = False

def on_connect(client, userdata, flags, rc):
    global mqtt_connected
    if rc == 0:
        mqtt_connected = True
        print("✅ Connected to MQTT Broker!")
    else:
        print(f"❌ Failed to connect, return code {rc}")

def on_disconnect(client, userdata, rc):
    global mqtt_connected
    mqtt_connected = False
    print(f"⚠️  Disconnected (rc={rc}), mencoba reconnect...")

client.on_connect    = on_connect
client.on_disconnect = on_disconnect

client.connect(BROKER, PORT, keepalive=60)
client.loop_start()

print("Menunggu koneksi MQTT...")
time.sleep(3)

# =====================================================
# MASTER NODE WANARA (25 NODE)
# =====================================================
nodes = [
    {"node_id": "node_1",  "forest_name": "Hutan Taman Nasional Gunung Leuser",          "latitude": 3.7580,  "longitude": 97.1300,  "district": "Ketambe",          "city": "Aceh Tenggara",      "province": "Aceh"},
    {"node_id": "node_2",  "forest_name": "Hutan Taman Nasional Kerinci Seblat",          "latitude": -1.6970, "longitude": 101.2640, "district": "Gunung Tujuh",     "city": "Kerinci",            "province": "Jambi"},
    {"node_id": "node_3",  "forest_name": "Hutan Taman Nasional Bukit Barisan Selatan",   "latitude": -5.3950, "longitude": 104.0330, "district": "Bengkunat",        "city": "Pesisir Barat",      "province": "Lampung"},
    {"node_id": "node_4",  "forest_name": "Hutan Taman Nasional Tanjung Puting",          "latitude": -2.8470, "longitude": 111.9650, "district": "Kumai",            "city": "Kotawaringin Barat", "province": "Kalimantan Tengah"},
    {"node_id": "node_5",  "forest_name": "Hutan Taman Nasional Kayan Mentarang",         "latitude": 3.5500,  "longitude": 116.6000, "district": "Mentarang",        "city": "Malinau",            "province": "Kalimantan Utara"},
    {"node_id": "node_6",  "forest_name": "Hutan Taman Nasional Lore Lindu",              "latitude": -1.3200, "longitude": 120.1800, "district": "Sigi Biromaru",    "city": "Sigi",               "province": "Sulawesi Tengah"},
    {"node_id": "node_7",  "forest_name": "Hutan Taman Nasional Aketajawe Lolobata",      "latitude": 1.1600,  "longitude": 128.0500, "district": "Wasile",           "city": "Halmahera Timur",    "province": "Maluku Utara"},
    {"node_id": "node_8",  "forest_name": "Hutan Taman Nasional Wasur",                   "latitude": -8.4200, "longitude": 140.9700, "district": "Merauke",          "city": "Merauke",            "province": "Papua Selatan"},
    {"node_id": "node_9",  "forest_name": "Hutan Taman Nasional Ujung Kulon",             "latitude": -6.7560, "longitude": 105.3370, "district": "Sumur",            "city": "Pandeglang",         "province": "Banten"},
    {"node_id": "node_10", "forest_name": "Hutan Taman Nasional Baluran",                 "latitude": -7.8430, "longitude": 114.3650, "district": "Banyuputih",       "city": "Situbondo",          "province": "Jawa Timur"},
    {"node_id": "node_11", "forest_name": "Hutan Taman Nasional Meru Betiri",             "latitude": -8.5470, "longitude": 113.8000, "district": "Tempurejo",        "city": "Jember",             "province": "Jawa Timur"},
    {"node_id": "node_12", "forest_name": "Hutan Taman Nasional Alas Purwo",              "latitude": -8.6500, "longitude": 114.3660, "district": "Tegaldlimo",       "city": "Banyuwangi",         "province": "Jawa Timur"},
    {"node_id": "node_13", "forest_name": "Hutan Taman Nasional Bromo Tengger Semeru",    "latitude": -8.1080, "longitude": 112.9200, "district": "Sukapura",         "city": "Probolinggo",        "province": "Jawa Timur"},
    {"node_id": "node_14", "forest_name": "Hutan Taman Nasional Sebangau",                "latitude": -2.3200, "longitude": 113.9200, "district": "Sebangau Kuala",   "city": "Pulang Pisau",       "province": "Kalimantan Tengah"},
    {"node_id": "node_15", "forest_name": "Hutan Taman Nasional Danau Sentarum",          "latitude": 0.8500,  "longitude": 112.0500, "district": "Batang Lupar",     "city": "Kapuas Hulu",        "province": "Kalimantan Barat"},
    {"node_id": "node_16", "forest_name": "Hutan Taman Nasional Kutai",                   "latitude": 0.5300,  "longitude": 117.4500, "district": "Teluk Pandan",     "city": "Kutai Timur",        "province": "Kalimantan Timur"},
    {"node_id": "node_17", "forest_name": "Hutan Taman Nasional Betung Kerihun",          "latitude": 0.9200,  "longitude": 112.1500, "district": "Putussibau Utara", "city": "Kapuas Hulu",        "province": "Kalimantan Barat"},
    {"node_id": "node_18", "forest_name": "Hutan Taman Nasional Bukit Baka Bukit Raya",   "latitude": -0.6500, "longitude": 112.3000, "district": "Serawai",          "city": "Sintang",            "province": "Kalimantan Barat"},
    {"node_id": "node_19", "forest_name": "Hutan Taman Nasional Bogani Nani Wartabone",   "latitude": 0.5400,  "longitude": 123.9200, "district": "Suwawa",           "city": "Bone Bolango",       "province": "Gorontalo"},
    {"node_id": "node_20", "forest_name": "Hutan Taman Nasional Rawa Aopa Watumohai",     "latitude": -4.4500, "longitude": 122.1000, "district": "Aopa",             "city": "Konawe Selatan",     "province": "Sulawesi Tenggara"},
    {"node_id": "node_21", "forest_name": "Hutan Taman Nasional Bantimurung Bulusaraung", "latitude": -4.8500, "longitude": 119.6800, "district": "Bantimurung",      "city": "Maros",              "province": "Sulawesi Selatan"},
    {"node_id": "node_22", "forest_name": "Hutan Taman Nasional Manusela",                "latitude": -3.0500, "longitude": 129.5200, "district": "Tehoru",           "city": "Maluku Tengah",      "province": "Maluku"},
    {"node_id": "node_23", "forest_name": "Hutan Taman Nasional Lorentz",                 "latitude": -4.7500, "longitude": 137.8300, "district": "Kwamki Narama",    "city": "Mimika",             "province": "Papua Tengah"},
    {"node_id": "node_24", "forest_name": "Hutan Taman Nasional Teluk Cenderawasih",      "latitude": -2.6500, "longitude": 134.5000, "district": "Rumberpon",        "city": "Teluk Wondama",      "province": "Papua Barat"},
    {"node_id": "node_25", "forest_name": "Hutan Taman Nasional Cycloop",                 "latitude": -2.5700, "longitude": 140.5200, "district": "Sentani Timur",    "city": "Jayapura",           "province": "Papua"},
]

node_status = {node["node_id"]: {"online": True, "offline_until": 0} for node in nodes}

# =====================================================
# AI ANOMALY DETECTION
# =====================================================
def calculate_anomaly_score(temp, humidity, smoke):
    norm_temp     = 30
    norm_humidity = 75
    norm_smoke    = 50
    temp_score     = abs(temp     - norm_temp)     / 20
    humidity_score = abs(humidity - norm_humidity) / 40
    smoke_score    = abs(smoke    - norm_smoke)    / 200
    return min(0.3 * temp_score + 0.2 * humidity_score + 0.5 * smoke_score, 1.0)

# =====================================================
# SENSOR DATA GENERATOR
# =====================================================
def generate_sensor_data(node):
    fire_detected = random.random() < 0.08

    if fire_detected:
        temperature = round(random.uniform(45, 65), 1)
        humidity    = round(random.uniform(15, 40), 1)
        smoke_level = random.randint(300, 800)
        status      = "FIRE_ALERT"
    else:
        temperature = round(random.uniform(24, 42), 1)
        humidity    = round(random.uniform(45, 90), 1)
        smoke_level = random.randint(10, 250)
        anomaly     = calculate_anomaly_score(temperature, humidity, smoke_level)
        status      = "WARNING" if anomaly >= 0.65 else "NORMAL"

    if random.random() < 0.06:
        signal   = random.randint(-100, -90)
        latency  = random.randint(300, 1200)
        loss     = round(random.uniform(5, 15), 2)
        n_status = "POOR"
    else:
        signal   = random.randint(-85, -55)
        latency  = random.randint(20, 150)
        loss     = round(random.uniform(0, 3), 2)
        n_status = "GOOD"

    return {
        "node_id":       node["node_id"],
        "node_name":     node["node_id"].replace("_", " ").title(),
        "forest_name":   node["forest_name"],
        "latitude":      node["latitude"],
        "longitude":     node["longitude"],
        "district":      node["district"],
        "city":          node["city"],
        "province":      node["province"],
        "temperature":   temperature,
        "humidity":      humidity,
        "fire_detected": fire_detected,
        "smoke_level":   smoke_level,
        "status":        status,
        "qos": {
            "network_status":      n_status,
            "signal_strength_dbm": signal,
            "latency_ms":          latency,
            "packet_loss_percent": loss
        },
        "timestamp": get_wib_time()
    }

# =====================================================
# MAIN LOOP
# =====================================================
print("=" * 60)
print("WANARA FOREST FIRE MONITORING SIMULATOR (STABLE MODE)")
print("=" * 60)
print(f"TOTAL NODE        : {len(nodes)}")
print(f"DELAY ANTAR NODE  : {DELAY_ANTAR_NODE}s")
print(f"JEDA ANTAR SIKLUS : {JEDA_ANTAR_SIKLUS}s")
print(f"TOTAL PER SIKLUS  : ~{int(len(nodes) * DELAY_ANTAR_NODE + JEDA_ANTAR_SIKLUS)}s (~30 detik)")
print(f"TIMEOUT BACKEND   : 120 detik (4 siklus toleransi)")
print("=" * 60)

siklus_ke = 0

try:
    while True:
        siklus_ke   += 1
        current_time = time.time()
        published    = 0
        skipped      = 0
        failed       = 0

        print(f"\n🔄 ── SIKLUS #{siklus_ke} ── {get_wib_time()} ──")

        for node in nodes:
            node_id = node["node_id"]
            ns      = node_status[node_id]

            # Cek apakah node sedang simulasi offline
            if not ns["online"]:
                if current_time >= ns["offline_until"]:
                    ns["online"] = True
                    print(f"   🔄 {node_id} kembali online")
                else:
                    sisa = int(ns["offline_until"] - current_time)
                    print(f"   💤 {node_id} masih offline ({sisa}s lagi)")
                    skipped += 1
                    time.sleep(DELAY_ANTAR_NODE)
                    continue

            # 1% chance simulasi node mati sementara
            if random.random() < 0.01:
                durasi              = random.randint(60, 180)
                ns["online"]        = False
                ns["offline_until"] = current_time + durasi
                print(f"   💤 {node_id} simulasi offline {durasi}s")
                skipped += 1
                time.sleep(DELAY_ANTAR_NODE)
                continue

            # Cek koneksi MQTT sebelum publish
            if not mqtt_connected:
                print(f"   ⚠️  MQTT terputus, menunggu reconnect...")
                time.sleep(3)
                if not mqtt_connected:
                    print(f"   ❌ Koneksi gagal, lewati siklus ini")
                    break

            payload = generate_sensor_data(node)

            try:
                result = client.publish(
                    f"wanara/{node_id}",
                    json.dumps(payload),
                    qos=1
                )
                # Tunggu konfirmasi broker maksimal 5 detik
                result.wait_for_publish(timeout=5)

                if result.is_published():
                    status_icon = "🔴" if payload["status"] == "FIRE_ALERT" else "🟡" if payload["status"] == "WARNING" else "🟢"
                    print(f"   {status_icon} {node_id:8s} | {payload['status']:10s} | {payload['temperature']}°C | {payload['smoke_level']} ppm")
                    published += 1
                else:
                    print(f"   ❌ {node_id} — broker tidak konfirmasi")
                    failed += 1

            except Exception as e:
                print(f"   ❌ {node_id} — error: {e}")
                failed += 1

            # Delay antar node — kunci utama agar broker tidak throttle
            time.sleep(DELAY_ANTAR_NODE)

        # Ringkasan siklus
        print(f"\n   📊 Siklus #{siklus_ke} selesai")
        print(f"      ✅ Terkirim : {published} node")
        print(f"      💤 Offline  : {skipped} node")
        print(f"      ❌ Gagal    : {failed} node")
        print(f"   ⏳ Jeda {JEDA_ANTAR_SIKLUS}s sebelum siklus berikutnya...")

        time.sleep(JEDA_ANTAR_SIKLUS)

except KeyboardInterrupt:
    print("\n🛑 Simulator dihentikan.")

finally:
    client.loop_stop()
    client.disconnect()